# TORAX + gyaradax-QL on the ITER hybrid scenario

Plugs the gyaradax JAX gyrokinetic solver into TORAX as a `QuasilinearTransportModel`. Runs ITER hybrid predictor-corrector (real CHEASE geometry, all 4 channels evolved) with `gyaradax-ql` and compares to the `qlknn` reference. Optional NL ground-truth at the bottom.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import sys, copy, time, dataclasses
ROOT = "/system/user/galletti/git/gyaradax"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import torax
from torax.examples import iterhybrid_predictor_corrector
from torax.plotting.configs import default_plot_config, transport_plot_config

from gyaradax.backends._cuda import is_available as _cuda_available
from gyaradax.torax_plugin import register_all
register_all()

# auto-pick the fastest available backend; jax is the fallback so the
# grad-smoke cell below still works on any machine.
BACKEND = "cuda" if _cuda_available() else "jax"
print(f"gyaradax backend: {BACKEND}")

## Calibrate Cn

Refit the Cn heads (scalar / parametric / polynomial) from the latest scan results in `data/cn_scan_partial/` and write `data/cn_iter_hybrid_<ts>.pkl`. The plug-in auto-discovers the newest pickle, so this just keeps the head fresh as more scan points trickle in. Fit-only (no NL re-run); takes seconds.

In [ ]:
import pickle, time as _t
sys.path.insert(0, f"{ROOT}/scripts")
import calibrate_torax_regime as cs

# fit on the aggregated GKW set (835 rows at native GKW res, eps=0.19,
# adiabatic electrons). Generalizes far better than the 54-pt gyaradax
# LHS — test R2 ~0.82 vs negative on the LHS-only split. eps coverage
# is fixed at this stage; the eps scan will broaden the LHS side later.
d = np.load(f"{ROOT}/data/aggregated_scan.npz", allow_pickle=True)
X_all, Y_all, F_all, src_all = d["X"], d["Y"], d["F"], d["source"]
src_kind = np.where(np.char.startswith(src_all.astype(str), "gkw"), "gkw", "gyaradax")
sel = np.isfinite(X_all) & np.isfinite(Y_all) & (X_all > 0) & (Y_all >= 0.1) & (src_kind == "gkw")
rows = [dict(X=float(X_all[i]), Y=float(Y_all[i]),
             F=np.asarray(F_all[i]), saturated=True)
        for i in np.where(sel)[0]]
print(f"GKW-only aggregated: {len(rows)} rows")

heads = cs.fit_heads(rows)
print(f"  scalar          : {heads['cn_scalar']:.4f}")
print(f"  parametric (d=1): R2_test={heads['r2_test']['parametric']:.4f}")
print(f"  polynomial_log  : R2_test={heads['r2_test']['polynomial_log']:.4f}")

ts = _t.strftime("%Y%m%d_%H%M%S")
out = f"{ROOT}/data/cn_iter_hybrid_{ts}.pkl"
with open(out, "wb") as f:
    pickle.dump({
        "scalar": heads["cn_scalar"],
        "parametric": heads["parametric"],
        "polynomial": heads["polynomial"],
        "polynomial_log": heads["polynomial_log"],
        "r2_test": heads["r2_test"], "r2_train": heads["r2_train"],
        "rmse_test": heads["rmse_test"],
        "n_train": heads["n_train"], "n_test": heads["n_test"],
        "source": "gkw_only", "n_rows": len(rows),
    }, f)
print(f"saved -> {out}")

# re-register so the plug-in's auto-discovery picks up the freshly written pkl
register_all()

## Scenario

Base case is TORAX's `iterhybrid_predictor_corrector` example: ITER hybrid at full size, real CHEASE equilibrium (R₀ = 6.2 m, a = 2.0 m, B₀ = 5.3 T, κ ≈ 1.8, δ ≈ 0.4 at edge), predictor-corrector linear solver with Pereverzev–Corrigan regularization. We swap the transport model from QLKNN to gyaradax-QL and freeze the channels we don't want to compare across. Everything else — equilibrium, sources, neoclassical/pedestal models, dt/CFL knobs, boundary-region patches — is the stock example, untouched.

The flux-tube physics inside gyaradax is local circular (Lapillonne `geom_type='circ'`), built from `(q, ŝ, ε)` extracted from the CHEASE equilibrium at each `rho_match`. Real shaping enters via the radial profile of those scalars but not via the local metric.

### Changes vs the stock example

| field | stock | this notebook |
|---|---|---|
| `transport.model_name` | `qlknn` | `gyaradax-ql` |
| `transport.backend` | — | `cuda` (auto, `jax` fallback) |
| `transport.rho_match` | (QLKNN evaluates every face) | 8 radii: `(0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)` |
| flux-tube grid | n/a | `(nvpar, nmu, ns, nkx, nky) = (32, 8, 16, 43, 16)`, `ikxspace=5` |
| `n_steps_linear` | n/a | 200 |
| `Cn` head | n/a | aggregated GKW fit (835 rows, test R² ≈ 0.82, parametric in `(ŝ, q, R/L_T, R/L_n)`) |
| `numerics.t_final` | 5.0 s | 1.0 s |
| `numerics.evolve_electron_heat` | True | **False** |
| `numerics.evolve_density` | True | **False** |
| `numerics.evolve_current` | True | **False** |

Everything not listed (geometry file, `numerics.{chi_timestep_prefactor, dt_reduction_factor, resistivity_multiplier, max_dt}`, `numerics.evolve_ion_heat`, all qlknn-style boundary patches `chi_min/max`, `D_e_min`, inner/outer chi patches, `rho_inner/outer`) is inherited verbatim from `iterhybrid_predictor_corrector.CONFIG`.

The reference QLKNN run uses the stock config with only `t_final` matched, so it's the same-evolution sanity check on the QL surrogate.

In [ ]:
T_FINAL_PHYS = 1.0      # full transport relaxation, ~10x the prior demo slice
RHO_MATCH = (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
RUN_NL = True           # ground-truth NL at one radius (slow; see NL cell)

ref_cfg_dict = copy.deepcopy(iterhybrid_predictor_corrector.CONFIG)
qlknn_transport = ref_cfg_dict["transport"]

cfg_dict = copy.deepcopy(ref_cfg_dict)
cfg_dict["numerics"]["evolve_ion_heat"] = True
cfg_dict["numerics"]["evolve_electron_heat"] = False
cfg_dict["numerics"]["evolve_density"] = False
cfg_dict["numerics"]["evolve_current"] = False
cfg_dict["numerics"]["t_final"] = T_FINAL_PHYS

cfg_dict["transport"] = {
    "model_name": "gyaradax-ql",
    "rho_match": RHO_MATCH,
    "backend": BACKEND,
    "n_steps_linear": 200,
    "nvpar": 32, "nmu": 8, "ns": 16, "nkx": 43, "nky": 16, "ikxspace": 5,
    "chi_min": qlknn_transport["chi_min"], "chi_max": qlknn_transport["chi_max"],
    "D_e_min": qlknn_transport["D_e_min"],
    "apply_inner_patch": qlknn_transport["apply_inner_patch"],
    "apply_outer_patch": qlknn_transport["apply_outer_patch"],
    "chi_i_inner": qlknn_transport["chi_i_inner"], "chi_e_inner": qlknn_transport["chi_e_inner"],
    "chi_i_outer": qlknn_transport["chi_i_outer"], "chi_e_outer": qlknn_transport["chi_e_outer"],
    "D_e_inner": qlknn_transport["D_e_inner"], "D_e_outer": qlknn_transport["D_e_outer"],
    "V_e_inner": qlknn_transport["V_e_inner"], "V_e_outer": qlknn_transport["V_e_outer"],
    "rho_inner": qlknn_transport["rho_inner"], "rho_outer": qlknn_transport["rho_outer"],
    "rho_min": qlknn_transport["rho_inner"], "rho_max": qlknn_transport["rho_outer"],
}

cfg = torax.ToraxConfig.from_dict(cfg_dict)
print(f"scenario:  ITER hybrid (CHEASE, T_i-only), t_final={T_FINAL_PHYS}s")
print(f"transport: {type(cfg.transport).__name__}, backend={BACKEND}")
print(f"rho_match: {cfg.transport.rho_match}  ({len(cfg.transport.rho_match)} radii)")
print(f"evolves:   T_i only (T_e, n_e, psi frozen at initial profile)")

## Run gyaradax-QL and QLKNN

Same scenario, two transport models. First JIT pass for gyaradax-QL is a few minutes; subsequent calls are cache hits.

In [ ]:
t0 = time.perf_counter()
data_gyaradax, _ = torax.run_simulation(cfg)
t_gyaradax = time.perf_counter() - t0

cfg_qlknn_dict = copy.deepcopy(ref_cfg_dict)
cfg_qlknn_dict["numerics"]["t_final"] = T_FINAL_PHYS
cfg_qlknn = torax.ToraxConfig.from_dict(cfg_qlknn_dict)
t0 = time.perf_counter()
data_qlknn, _ = torax.run_simulation(cfg_qlknn)
t_qlknn = time.perf_counter() - t0

Q_g = float(data_gyaradax.scalars.Q_fusion.values[-1])
Q_q = float(data_qlknn.scalars.Q_fusion.values[-1])
print(f"gyaradax-ql  {t_gyaradax:6.1f}s   Q_fusion = {Q_g:.3f}")
print(f"qlknn        {t_qlknn:6.1f}s   Q_fusion = {Q_q:.3f}   (Δ = {100*(Q_g-Q_q)/Q_q:+.1f}%)")

## TORAX 4×4 panel — gyaradax (solid) overlaid on QLKNN (dashed)

In [ ]:
fig = torax.plot_run_from_data_tree(
    plot_config=default_plot_config.PLOT_CONFIG,
    data_tree=data_gyaradax, data_tree2=data_qlknn,
    interactive=False,
    fig_title="ITER hybrid: gyaradax-QL vs QLKNN",
)
fig.show()

## Transport coefficients

In [ ]:
fig = torax.plot_run_from_data_tree(
    plot_config=transport_plot_config.PLOT_CONFIG,
    data_tree=data_gyaradax, data_tree2=data_qlknn,
    interactive=False,
    fig_title="chi / D / V profiles",
)
fig.show()

## Per-radius diagnostic

Standalone QL solve at one representative `(rlt, rln, q, shat, eps)` — γ(ky), gate, sat_amp.

In [ ]:
from gyaradax.params import GKParams
from gyaradax.torax_plugin.gyaradax_based_transport_model import gyaradax_geometry_at
from gyaradax.torax_plugin.gyaradax_ql_transport_model import GyaradaxQLTransportModel
from gyaradax.solver import gksolve, linear_precompute, default_state
from gyaradax.integrals import calculate_fluxes, geom_tensors
from gyaradax.quasilinear.saturation import ql_flux_diagnostics

tm = GyaradaxQLTransportModel.from_config(cfg.transport)
params = dataclasses.replace(
    GKParams(),
    rlt=jnp.asarray(8.0), rln=jnp.asarray(2.5),
    q=jnp.asarray(2.0), shat=jnp.asarray(1.0), eps=jnp.asarray(0.18),
    beta=jnp.asarray(0.0),
    adiabatic_electrons=True, non_linear=False, disable_per_ky_norm=True,
    dt=0.005,
)
geom_at_r = gyaradax_geometry_at(
    q=params.q, shat=params.shat, eps=params.eps,
    config=cfg.transport, topology=tm.topology,
)
df_f, (phi, _), state_f = gksolve(
    tm._initial_df(), geom_at_r, params,
    default_state(nky=cfg.transport.nky),
    n_steps=cfg.transport.n_steps_linear,
    pre=linear_precompute(geom_at_r, params),
)
df_f.block_until_ready()

gt = geom_tensors(geom_at_r)
_, eflux_kxy, _ = calculate_fluxes(gt, df_f, phi, reduce=False)
ints = jnp.asarray(geom_at_r["ints"])
phi2 = jnp.abs(phi) ** 2
phi2_kxy = jnp.sum(phi2 * ints[:, None, None], axis=0)
lg = jnp.asarray(geom_at_r["little_g"])
diag = ql_flux_diagnostics(
    growth_rate=state_f.last_growth_rate,
    phi2=phi2, phi2_kxy=phi2_kxy, flux_kxy=eflux_kxy,
    krho=jnp.asarray(geom_at_r["krho"]),
    kxrh=jnp.asarray(geom_at_r["kxrh"]),
    little_g=lg if lg.shape[0] == 3 else lg.T,
    ds=jnp.mean(ints),
)
print("gamma(ky) =", np.asarray(state_f.last_growth_rate))
print("Q_QL      =", float(diag["Q_QL"]))

## Optional ground-truth: gyaradax-NL

Set `RUN_NL = True` in the config cell. Each transport call runs a full NL gyaradax sim; tens of minutes per TORAX dt, so the NL run uses a single radius and a very short `t_final`.

In [ ]:
data_nl = None
if RUN_NL:
    # NL pydantic model only accepts the shared base fields + NL-specific
    # ones (mode, n_steps_nl, n_average, flux_table_path). Strip QL-only
    # fields and the (qlknn-style) inner/outer patches before validating.
    QL_ONLY = {"n_steps_linear", "ncv_eigensolve", "cn_calibration_path",
               "cn_scalar"}
    PATCH_KEYS = {"chi_min", "chi_max", "D_e_min",
                  "apply_inner_patch", "apply_outer_patch",
                  "chi_i_inner", "chi_e_inner", "chi_i_outer", "chi_e_outer",
                  "D_e_inner", "D_e_outer", "V_e_inner", "V_e_outer",
                  "rho_inner", "rho_outer", "rho_min", "rho_max"}
    nl_t = {k: v for k, v in cfg_dict["transport"].items()
            if k not in QL_ONLY | PATCH_KEYS}
    nl_t["model_name"] = "gyaradax-nl"
    nl_t["mode"] = "direct"
    nl_t["backend"] = BACKEND           # cuda when available (~10x faster NL)
    nl_t["rho_match"] = (0.2, 0.4, 0.6, 0.8)
    nl_t["n_steps_nl"] = 20_000
    nl_t["n_average"] = 2000

    cfg_nl_dict = copy.deepcopy(cfg_dict)
    cfg_nl_dict["transport"] = nl_t
    cfg_nl_dict["numerics"]["t_final"] = 0.1

    cfg_nl = torax.ToraxConfig.from_dict(cfg_nl_dict)
    t0 = time.perf_counter()
    data_nl, _ = torax.run_simulation(cfg_nl)
    print(f"gyaradax-nl  {(time.perf_counter()-t0)/60:.1f}min   "
          f"Q_fusion = {float(data_nl.scalars.Q_fusion.values[-1]):.3f}")
else:
    print("NL disabled (RUN_NL=False).")

## Three-way comparison

In [ ]:
rho_face = data_gyaradax.profiles.rho_face_norm.values
rho_cell = data_gyaradax.profiles.rho_norm.values

def _last(d, key):
    return getattr(d.profiles, key).values[-1]

fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
rows = [
    ("chi_turb_i", r"$\chi_{i}$ [m$^2$/s]", rho_face),
    ("T_i", r"$T_i$ [keV]", rho_cell),
    ("T_e", r"$T_e$ [keV]", rho_cell),
    ("n_e", r"$n_e$ [$10^{20}$/m$^3$]", rho_cell),
]
for ax, (key, ylabel, x) in zip(axes.ravel(), rows):
    ax.plot(x, _last(data_qlknn, key), "C0-", lw=2, label="qlknn")
    ax.plot(x, _last(data_gyaradax, key), "C1--", lw=2, label="gyaradax-ql")
    if data_nl is not None:
        ax.plot(x, _last(data_nl, key), "C3:", lw=2, label="gyaradax-nl")
    ax.set_xlabel(r"$\rho_{norm}$"); ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

for label, d in [("qlknn", data_qlknn), ("gyaradax-ql", data_gyaradax),
                 ("gyaradax-nl", data_nl)]:
    if d is None:
        continue
    print(f"{label:13s}  Q_fusion={float(d.scalars.Q_fusion.values[-1]):6.3f}  "
          f"tau_E={float(d.scalars.tau_E.values[-1]):.4f}")

## Gradients

With `backend="jax"`, gyaradax's geometry refactor (`build_topology` + `compute_continuous_geometry`) makes the path through gyaradax pure JAX, so `jax.grad` propagates end-to-end through TORAX.

Concrete uses, in roughly increasing project value:
1. Sensitivity analysis (`jax.grad(Q_fusion)(rlt_at_rho)`) — replaces finite-difference noise.
2. Calibrate `C_n` against experimental / QLKNN targets via Adam on `(Q_torax − Q_target)²`.
3. Pulse-design optimization on `Ip(t)` / heating waveforms via `experimental.run_loop_jit`.
4. PORTALS-style flux matching with AD-enhanced GP kernels (Jacobian-augmented acquisition).
5. Direct gradient-only flux matching, skipping the GP entirely on the QL path.

Smoke test below: `jax.grad(Q_i)(rlt)` at one radius.

In [ ]:
def qi_of_rlt(rlt):
    p = dataclasses.replace(
        GKParams(),
        rlt=rlt, rln=jnp.asarray(2.5), q=jnp.asarray(2.0),
        shat=jnp.asarray(1.0), eps=jnp.asarray(0.18), beta=jnp.asarray(0.0),
        adiabatic_electrons=True, non_linear=False,
        disable_per_ky_norm=True, dt=0.005,
    )
    g = gyaradax_geometry_at(q=p.q, shat=p.shat, eps=p.eps,
                              config=cfg.transport, topology=tm.topology)
    qi, _, _ = tm._gyaradax_ql_at_radius(p, g)
    return qi

g = float(jax.jit(jax.grad(qi_of_rlt))(jnp.asarray(8.0)))
print(f"d Q_i / d rlt  =  {g:+.4e}    (finite, non-zero, positive — AD path is alive)")

## Headline figure

Single 2×2 summary: T_i and chi_i at t_final, Q_fusion(t), and the technical summary box.

In [ ]:
rho_face = data_gyaradax.profiles.rho_face_norm.values
rho_cell = data_gyaradax.profiles.rho_norm.values
t_g = data_gyaradax.scalars.time.values
t_q = data_qlknn.scalars.time.values
t_n = data_nl.scalars.time.values if data_nl is not None else None

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))

ax = axes[0, 0]
ax.plot(rho_cell, data_qlknn.profiles.T_i.values[-1], "C0-", lw=2.4, label="TORAX + QLKNN")
ax.plot(rho_cell, data_gyaradax.profiles.T_i.values[-1], "C3--", lw=2.4, label="TORAX + gyaradax-QL")
if data_nl is not None:
    ax.plot(rho_cell, data_nl.profiles.T_i.values[-1], "C2:", lw=2.4, label="TORAX + gyaradax-NL")
ax.set_xlabel(r"$\rho_{norm}$"); ax.set_ylabel(r"$T_i$ [keV]")
ax.set_title("(a)  Ion temperature", fontsize=11)
ax.grid(alpha=0.3); ax.legend(fontsize=10)

ax = axes[0, 1]
ax.plot(rho_face, data_qlknn.profiles.chi_turb_i.values[-1], "C0-", lw=2.4)
ax.plot(rho_face, data_gyaradax.profiles.chi_turb_i.values[-1], "C3--", lw=2.4)
if data_nl is not None:
    ax.plot(rho_face, data_nl.profiles.chi_turb_i.values[-1], "C2:", lw=2.4)
ax.set_xlabel(r"$\rho_{norm}$"); ax.set_ylabel(r"$\chi_{turb,i}$ [m$^2$/s]")
ax.set_title("(b)  Ion heat diffusivity", fontsize=11)
ax.grid(alpha=0.3)

ax = axes[1, 0]
ax.plot(t_q, data_qlknn.scalars.Q_fusion.values, "C0-o", lw=2.4, ms=5)
ax.plot(t_g, data_gyaradax.scalars.Q_fusion.values, "C3--s", lw=2.4, ms=5)
if data_nl is not None:
    ax.plot(t_n, data_nl.scalars.Q_fusion.values, "C2:^", lw=2.4, ms=5)
ax.set_xlabel("t [s]"); ax.set_ylabel(r"$Q_{fusion}$")
ax.set_title("(c)  Fusion gain $Q_{fusion}(t)$", fontsize=11)
ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.axis("off")
qf_g = float(data_gyaradax.scalars.Q_fusion.values[-1])
qf_q = float(data_qlknn.scalars.Q_fusion.values[-1])
tau_g = float(data_gyaradax.scalars.tau_E.values[-1])
tau_q = float(data_qlknn.scalars.tau_E.values[-1])
summary = (
    f"Scenario:      ITER hybrid (predictor-corrector, CHEASE)\n"
    f"Evolves:       T_i only (T_e, n_e, psi frozen)\n"
    f"t_final:       {T_FINAL_PHYS}s\n"
    f"\n"
    f"Q_fusion       QLKNN       = {qf_q:.3f}\n"
    f"               gyaradax-QL = {qf_g:.3f}  ({100*(qf_g-qf_q)/qf_q:+.1f}%)\n"
)
if data_nl is not None:
    qf_n = float(data_nl.scalars.Q_fusion.values[-1])
    tau_n = float(data_nl.scalars.tau_E.values[-1])
    summary += (
        f"               gyaradax-NL = {qf_n:.3f}  ({100*(qf_n-qf_q)/qf_q:+.1f}%)\n"
    )
summary += (
    f"\n"
    f"tau_E          QLKNN       = {tau_q:.4f}\n"
    f"               gyaradax-QL = {tau_g:.4f}  ({100*(tau_g-tau_q)/tau_q:+.1f}%)\n"
)
if data_nl is not None:
    summary += (
        f"               gyaradax-NL = {tau_n:.4f}  ({100*(tau_n-tau_q)/tau_q:+.1f}%)\n"
    )
summary += (
    f"\n"
    f"Wallclock      QLKNN       {t_qlknn:6.1f}s\n"
    f"               gyaradax-QL {t_gyaradax:6.1f}s\n"
)
ax.text(0.0, 1.0, summary, transform=ax.transAxes,
        ha="left", va="top", family="monospace", fontsize=9)

fig.suptitle("gyaradax × TORAX — JAX gyrokinetic transport plug-in", fontsize=13, y=1.00)
plt.tight_layout()
fig.savefig("torax_gyaradax_headline.png", dpi=160, bbox_inches="tight")
plt.show()